# 🐐 FarmGuard — Animal Heat-Risk Classification Model

**Goal:** Classify livestock heat-stress risk (`Low`, `Moderate`, `High`, `Critical`) from animal
characteristics and environmental conditions, so farmers get an early warning before heat stress
harms their animals.

| | |
|---|---|
| **Dataset** | `farmguard_animal_heat_risk.csv` (10,000 records, 3 species) |
| **Target** | `risk_level` — Low / Moderate / High / Critical |
| **Model** | Random Forest (class-balanced, 300 trees) |
| **Species covered** | Goat, Sheep, Cattle (15 breeds total) |

**Key features**
- Animal traits: species, breed, sex, age, weight, physiological stage (lactating/growing/pregnant/dry)
- Environment: temperature, humidity, location (lat/long)
- Derived heat-stress indices: **THI** (Temperature-Humidity Index) and **HLI** (Heat Load Index),
  precomputed in the source dataset

---


## 1. Setup & Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

print("Libraries loaded successfully")

## 2. Load Dataset

The dataset can either be uploaded directly to the Colab session or read from Google Drive.
By default this notebook mounts Drive (useful for repeated runs); if the CSV is instead sitting
next to the notebook (e.g. uploaded via the Colab file browser), it will be used automatically.

In [ ]:
DATA_FILENAME = "farmguard_animal_heat_risk.csv"

# Try a local copy first (e.g. uploaded directly into the Colab session)
if os.path.exists(DATA_FILENAME):
    dataset_path = DATA_FILENAME
else:
    from google.colab import drive
    drive.mount('/content/drive')
    dataset_path = f"/content/drive/MyDrive/{DATA_FILENAME}"

df = pd.read_csv(dataset_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

In [ ]:
print("Risk level distribution:")
print(df['risk_level'].value_counts())

print("\nSpecies distribution:")
print(df['species'].value_counts())

df.head()

## 3. Exploratory Data Analysis (EDA)

Quick look at class balance, how THI/HLI relate to the target, and correlations between numeric
features before deciding on preprocessing steps.

In [ ]:
# Class imbalance: "Low" risk dominates, "Critical" and "High" are much rarer
# but the most important classes for the model to catch.
plt.figure(figsize=(7, 4))
order = ['Low', 'Moderate', 'High', 'Critical']
sns.countplot(data=df, x='risk_level', order=order, palette='YlOrRd')
plt.title('Animal Heat Risk Level Distribution')
plt.ylabel('Number of records')
plt.xlabel('Risk Level')
plt.tight_layout()
plt.show()

In [ ]:
# THI (Temperature-Humidity Index) is expected to be the strongest predictor —
# check how it separates the risk classes.
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='risk_level', y='thi', order=order, palette='YlOrRd')
plt.title('THI (Temperature-Humidity Index) by Risk Level')
plt.xlabel('Risk Level')
plt.ylabel('THI')
plt.tight_layout()
plt.show()

In [ ]:
# Species mix within each risk level — do some species show up as higher-risk more often?
plt.figure(figsize=(8, 5))
species_risk = pd.crosstab(df['species'], df['risk_level'], normalize='index')[order]
species_risk.plot(kind='bar', stacked=True, colormap='YlOrRd', figsize=(8, 5))
plt.title('Risk Level Composition by Species')
plt.ylabel('Proportion')
plt.xlabel('Species')
plt.legend(title='Risk Level', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between numeric features
numeric_cols = [
    'age_years', 'weight_kg', 'latitude', 'longitude',
    'temperature_c', 'humidity_percent', 'thi', 'hli',
]

plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Between Numeric Features')
plt.tight_layout()
plt.show()

## 4. Define Features & Target

In [ ]:
categorical_features = [
    "species",
    "breed",
    "sex",
    "physiological_stage",
]

numerical_features = [
    "age_years",
    "weight_kg",
    "latitude",
    "longitude",
    "temperature_c",
    "humidity_percent",
    "thi",
    "hli",
]

target = "risk_level"
features_to_use = categorical_features + numerical_features

# Sanity check: make sure every expected column actually exists in the dataset
required_columns = features_to_use + [target]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f" Missing columns in dataset: {missing_columns}")

print(" Features selected successfully")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"\nTarget: {target}")

## 5. Data Cleaning & Preprocessing

Checks for missing values, then label-encodes the four categorical columns (`species`, `breed`,
`sex`, `physiological_stage`) and the target — required before feeding data to a scikit-learn model.

**Note on THI/HLI:** both are already present in the source dataset (precomputed), so this notebook
uses them as-is rather than recalculating them. Any service that serves this model in production
must recompute THI/HLI using the *exact same formulas* used to build this dataset, or predictions
will be inconsistent with what the model learned (see Section 10 for details).

In [ ]:
missing_values = df[required_columns].isnull().sum()
print("Missing values per required column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "  None found ")

if missing_values.sum() > 0:
    print("\n Removing rows with missing values...")
    df = df.dropna(subset=required_columns).copy()

print(f"\nDataset shape after cleaning: {df.shape}")

In [ ]:
# Encode categorical input features
label_encoders = {}

for col in categorical_features:
    le = LabelEncoder()
    df[col + "_encoded"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"{col}: {list(le.classes_)}")

print("\n Categorical features encoded successfully")

In [ ]:
# Encode the target label
target_encoder = LabelEncoder()
df["risk_level_encoded"] = target_encoder.fit_transform(df[target].astype(str))

print("Target classes:", list(target_encoder.classes_))
print("\nEncoding map:")
for class_name, encoded_value in zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)):
    print(f"  {class_name} → {encoded_value}")

In [ ]:
# Final feature matrix (encoded categoricals + raw numerical features)
feature_cols = [col + "_encoded" for col in categorical_features] + numerical_features

X = df[feature_cols].copy()
y = df["risk_level_encoded"].copy()

print(f"Number of features: {len(feature_cols)}")
print(f"Number of samples: {len(X)}")
print("\nFinal model features:")
for i, feature in enumerate(feature_cols, start=1):
    print(f"  {i}. {feature}")

## 6. Model Building

We split the data into train/test sets, scale the numerical features (Random Forest doesn't
strictly need this, but it's kept for consistency with the production API's preprocessing), and
train a class-balanced Random Forest.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(" Dataset split successfully")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

In [ ]:
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print(" Numerical features scaled successfully")

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",   # handles the Low-class majority automatically
    random_state=42,
    n_jobs=-1,
)

rf_model.fit(X_train_scaled, y_train)

print(" Model trained successfully")

## 7. Model Evaluation

In [ ]:
y_pred = rf_model.predict(X_test_scaled)
y_pred_proba = rf_model.predict_proba(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 50)
print(f"Accuracy:        {accuracy:.4f}")
print(f"Macro F1 Score:    {macro_f1:.4f}")
print(f"Weighted F1 Score: {weighted_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_encoder.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_,
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Animal Heat Risk — Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance')
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance)

## 8. Save & Reload the Model

The trained model plus every artifact needed to reproduce its preprocessing (scaler, encoders,
feature order) are saved together so the model can be reloaded and used independently of this
notebook — this is exactly what the Flask API in `app.py` loads at startup.

In [ ]:
save_path = '/content/drive/MyDrive/models' if 'drive' in dir() else './models'
os.makedirs(save_path, exist_ok=True)

joblib.dump(rf_model, f'{save_path}/animal_heat_risk_model.pkl')
joblib.dump(scaler, f'{save_path}/feature_scaler.pkl')
joblib.dump(label_encoders, f'{save_path}/label_encoders.pkl')
joblib.dump(target_encoder, f'{save_path}/target_encoder.pkl')
joblib.dump(feature_cols, f'{save_path}/feature_columns.pkl')

print(" Model artifacts saved successfully!")
print(f"\n Save location: {save_path}")
print("\nSaved files:")
print("  - animal_heat_risk_model.pkl")
print("  - feature_scaler.pkl")
print("  - label_encoders.pkl")
print("  - target_encoder.pkl")
print("  - feature_columns.pkl")

In [ ]:
loaded_model = joblib.load(f'{save_path}/animal_heat_risk_model.pkl')
loaded_scaler = joblib.load(f'{save_path}/feature_scaler.pkl')
loaded_label_encoders = joblib.load(f'{save_path}/label_encoders.pkl')
loaded_target_encoder = joblib.load(f'{save_path}/target_encoder.pkl')
loaded_feature_columns = joblib.load(f'{save_path}/feature_columns.pkl')

print(" Model and supporting files loaded successfully!")
print("\nLoaded feature order:")
print(loaded_feature_columns)

In [ ]:
# Sanity check: predict on a held-out sample and compare to the true label
sample = X_test.iloc[[0]].copy()
sample_scaled = sample.copy()
sample_scaled[numerical_features] = loaded_scaler.transform(sample[numerical_features])

prediction_encoded = loaded_model.predict(sample_scaled)
prediction = loaded_target_encoder.inverse_transform(prediction_encoded)[0]
actual = loaded_target_encoder.inverse_transform([y_test.iloc[0]])[0]
probabilities = loaded_model.predict_proba(sample_scaled)[0]

print(f" Predicted risk level: {prediction}")
print(f" Actual risk level:    {actual}")
print("\nProbabilities:")
for class_name, probability in zip(loaded_target_encoder.classes_, probabilities):
    print(f"  {class_name}: {probability:.2%}")

## 9. Prediction Function

A single reusable function that takes raw animal + environmental data, computes THI/HLI, and
returns the predicted risk level plus class probabilities — mirrors the logic in the Flask API.

In [ ]:
def calculate_thi(temperature_c, humidity_percent):
    """Temperature-Humidity Index — must match the formula used to build the training data."""
    return (1.8 * temperature_c + 32) - (
        (0.55 - 0.0055 * humidity_percent) * (1.8 * temperature_c - 26)
    )


def calculate_hli(temperature_c, humidity_percent):
    """
    Heat Load Index.

    ⚠️ IMPORTANT: the exact HLI formula used to generate this dataset's `hli` column is not
    reproduced by this simplified placeholder. Before using this in production, derive the real
    formula (e.g. from the data source / domain literature) and update this function — otherwise
    predictions will be based on an HLI the model was never actually trained on.
    """
    return temperature_c + (0.33 * humidity_percent / 100)


def predict_animal_heat_risk(new_data):
    """
    Predict animal heat risk from new input data.

    Parameters
    ----------
    new_data : pd.DataFrame
        Must contain the raw columns: species, breed, sex, physiological_stage,
        age_years, weight_kg, latitude, longitude, temperature_c, humidity_percent
        (thi and hli are computed automatically).

    Returns
    -------
    predicted_classes : np.ndarray
        Predicted risk_level labels (e.g. "Low", "Critical").
    probabilities : np.ndarray
        Class probabilities for each row, in target_encoder.classes_ order.
    """
    new_data = new_data.copy()

    model = joblib.load(f'{save_path}/animal_heat_risk_model.pkl')
    scaler = joblib.load(f'{save_path}/feature_scaler.pkl')
    label_encoders = joblib.load(f'{save_path}/label_encoders.pkl')
    target_encoder = joblib.load(f'{save_path}/target_encoder.pkl')
    feature_columns = joblib.load(f'{save_path}/feature_columns.pkl')

    required_cols = [
        'species', 'breed', 'sex', 'physiological_stage', 'age_years', 'weight_kg',
        'latitude', 'longitude', 'temperature_c', 'humidity_percent',
    ]
    missing_cols = [c for c in required_cols if c not in new_data.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    new_data['thi'] = calculate_thi(new_data['temperature_c'], new_data['humidity_percent'])
    new_data['hli'] = calculate_hli(new_data['temperature_c'], new_data['humidity_percent'])

    for col, encoder in label_encoders.items():
        values = new_data[col].astype(str)
        unknown_values = set(values) - set(encoder.classes_)
        if unknown_values:
            raise ValueError(f"Unknown values for '{col}': {list(unknown_values)}")
        new_data[col + '_encoded'] = encoder.transform(values)

    X_new = new_data[feature_columns].copy()
    X_new[numerical_features] = scaler.transform(X_new[numerical_features])

    predictions = model.predict(X_new)
    probabilities = model.predict_proba(X_new)
    predicted_classes = target_encoder.inverse_transform(predictions)

    return predicted_classes, probabilities


print(" Prediction function ready!")

In [ ]:
# Example: predict heat risk for a single new animal
new_data = pd.DataFrame([{
    'species': 'cattle',
    'breed': 'Holstein',
    'sex': 'female',
    'physiological_stage': 'lactating',
    'age_years': 4.0,
    'weight_kg': 620.0,
    'latitude': 30.1,
    'longitude': -8.8,
    'temperature_c': 36.5,
    'humidity_percent': 55,
}])

predicted_classes, probabilities = predict_animal_heat_risk(new_data)

print("=" * 50)
print("🐄 ANIMAL HEAT RISK PREDICTION")
print("=" * 50)
print(f"\nPredicted Risk: {predicted_classes[0]}")
print("\nPrediction Probabilities:")
for class_name, probability in zip(target_encoder.classes_, probabilities[0]):
    print(f"  {class_name}: {probability:.2%}")

## 10. Summary & Production Deployment Checklist

**Dataset**
- 10,000 animal records across 3 species (goat, sheep, cattle) and 15 breeds
- 10 raw input fields + 2 derived heat-stress indices (THI, HLI)
- Target `risk_level` is imbalanced: Low 6,150 · Moderate 1,575 · Critical 1,261 · High 1,014

**Model**
- Random Forest (300 trees, `class_weight="balanced"`) trained on an 80/20 stratified split
- Numerical features are standard-scaled; categorical features are label-encoded


**Status**
- ✅ Trained on real animal + environmental data across 3 species
- ✅ Class imbalance handled via `class_weight="balanced"`
- ✅ Reusable `predict_animal_heat_risk()` inference function


**Saved model files**
- `animal_heat_risk_model.pkl` — trained Random Forest model
- `feature_scaler.pkl` — StandardScaler fitted on numerical features
- `feature_columns.pkl` — exact feature column order used at training time
- `label_encoders.pkl` — categorical feature encoders
- `target_encoder.pkl` — target label encoder

**Usage**
1. Run all cells above once to train and save the model artifacts.
2. In any future session, load the saved artifacts and call `predict_animal_heat_risk(new_data)`
   with a DataFrame matching the required raw columns (see function docstring in Section 9).
3. Recommended next step: resolve the HLI formula mismatch, then re-validate model accuracy.
